In [1]:
import os
print(os.getcwd())

/Users/siqinling/medrag_project/notebooks


In [2]:
import pandas as pd
df = pd.read_parquet('../data/processed/chunks.parquet')
print(df.columns.tolist())
print(df.shape)
df.head(2)

['chunk_id', 'text', 'doc_id', 'chunk_index', 'total_chunks', 'source_title', 'token_count', 'pmid', 'journal', 'pub_date']
(170237, 10)


,chunk_id,text,doc_id,chunk_index,total_chunks,source_title,token_count,pmid,journal,pub_date
0,PMC524367,"Perceived personal, social and environmental b...",PMC524367,0,1,"Perceived personal, social and environmental b...",296,15462679,The International Journal of Behavioral Nutrit...,2004-10-5
1,PMC545955_chunk0,Antitumor effects of two bisdioxopiperazines a...,PMC545955,0,2,Antitumor effects of two bisdioxopiperazines a...,384,15617579,BMC Pharmacology,2004-12-24


In [3]:
import sys
sys.path.append('../scripts')
from importlib import import_module
qp = import_module('04_query_processing')

query_info = qp.process_medical_query("二甲双胍对心血管疾病有何影响？")
print(query_info.keys())
print(query_info)

dict_keys(['original_query', 'cleaned_query', 'entities', 'expanded_terms', 'vector_query', 'keyword_query', 'filters'])
{'original_query': '二甲双胍对心血管疾病有何影响？', 'cleaned_query': '二甲双胍对心血管疾病有何影响?', 'entities': {'drug': ['二甲双胍']}, 'expanded_terms': ['metformin'], 'vector_query': 'Represent this question for searching relevant passages: 二甲双胍对心血管疾病有何影响?', 'keyword_query': '二甲双胍对心血管疾病有何影响? metformin', 'filters': {}}


In [4]:
sys.path.append('../scripts')
mpr_module = import_module('05_multipath_retrieval')

retriever = mpr_module.MultiPathRetriever(
    chunks_path='../data/processed/chunks.parquet',
    chroma_db_path='../data/processed/chroma_db'
)


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1431.81it/s]
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/y5/4900bss12yg8nhh_4m3wz5q00000gn/T/jieba.cache


构建BM25索引...


Loading model cost 0.264 seconds.
Prefix dict has been built successfully.


In [5]:
vector_results = retriever.vector_search(query_info, top_k=5)
keyword_results = retriever.keyword_search(query_info, top_k=5)

print("=== 向量检索结果 ===")
for r in vector_results:
    print(r['rank'], r['chunk_id'], round(r['score'], 3), r['text'][:50])

print("\n=== BM25检索结果 ===")
for r in keyword_results:
    print(r['rank'], r['chunk_id'], round(r['score'], 3), r['text'][:50])


=== 向量检索结果 ===
1 PMC2958157_chunk0 0.677 The ChQoL questionnaire: an Italian translation wi
2 PMC2939644 0.667 Three versions of Perceived Stress Scale: validati
3 PMC2500013 0.665 Concordance between Hopkins Symptom Checklist (HSC
4 PMC1434784 0.663 Measuring changes in self-concept: a qualitative e
5 PMC1360658 0.662 Structural ambiguity of the Chinese version of the

=== BM25检索结果 ===
1 PMC2694802 24.067 Does metformin affect ovarian morphology in patien
2 PMC2847989 22.549 Assessment of efficacy and tolerability of once-da
3 PMC2807458 22.235 Metformin Induces a Dietary Restriction–Like State
4 PMC2613390_chunk0 21.819 Cell cycle arrest in Metformin treated breast canc
5 PMC1974811_chunk0 21.81 Rosiglitazone RECORD study: glucose control outcom


In [6]:
import importlib
importlib.reload(mpr_module)

retriever = mpr_module.MultiPathRetriever(
    chunks_path='../data/processed/chunks.parquet',
    chroma_db_path='../data/processed/chroma_db'
)

vector_results = retriever.vector_search(query_info, top_k=5)
for r in vector_results:
    print(r['rank'], r['chunk_id'], round(r['score'], 3), r['text'][:50])


加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1419.99it/s]


构建BM25索引...
1 PMC2958157_chunk0 0.677 The ChQoL questionnaire: an Italian translation wi
2 PMC2939644 0.667 Three versions of Perceived Stress Scale: validati
3 PMC2500013 0.665 Concordance between Hopkins Symptom Checklist (HSC
4 PMC1434784 0.663 Measuring changes in self-concept: a qualitative e
5 PMC1360658 0.662 Structural ambiguity of the Chinese version of the


In [7]:
test_query_info_en = {
    'cleaned_query': 'What is the effect of metformin on cardiovascular disease?',
    'keyword_query': 'metformin cardiovascular disease',
}

vector_results_en = retriever.vector_search(test_query_info_en, top_k=5)
for r in vector_results_en:
    print(r['rank'], r['chunk_id'], round(r['score'], 3), r['text'][:50])


1 PMC2644685 0.794 Metformin treatment in diabetes and heart failure:
2 PMC2991324 0.791 Long-term effect of metformin on blood glucose con
3 PMC2946277_chunk0 0.789 Effects of oral glucose-lowering drugs on long ter
4 PMC2705820 0.776 Antihypertensive therapy, new-onset diabetes, and 
5 PMC2723076_chunk0 0.764 A cardiologic approach to non-insulin antidiabetic


In [8]:
importlib.reload(mpr_module)
retriever = mpr_module.MultiPathRetriever(
    chunks_path='../data/processed/chunks.parquet',
    chroma_db_path='../data/processed/chroma_db'
)

query_info_en = qp.process_medical_query("What is the effect of metformin on cardiovascular disease?")

for strategy in ['simple', 'rrf', 'weighted']:
    print(f"\n=== {strategy} ===")
    results = retriever.fusion_search(query_info_en, fusion_strategy=strategy, top_k_final=5)
    for r in results:
        print(r['rank'], r['chunk_id'], round(r['score'], 4), r['text'][:50])


加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1081.03it/s]


构建BM25索引...

=== simple ===
1 PMC2644685 0.7944 Metformin treatment in diabetes and heart failure:
2 PMC2991324 0.7908 Long-term effect of metformin on blood glucose con
3 PMC2946277_chunk0 0.7893 Effects of oral glucose-lowering drugs on long ter
4 PMC2705820 0.7759 Antihypertensive therapy, new-onset diabetes, and 
5 PMC2723076_chunk0 0.7641 A cardiologic approach to non-insulin antidiabetic

=== rrf ===
1 PMC2644685 0.0164 Metformin treatment in diabetes and heart failure:
2 PMC2875210 0.0164 Using family history information to promote health
3 PMC2991324 0.0161 Long-term effect of metformin on blood glucose con
4 PMC2729049 0.0161 Peroxisome Proliferator-Activated Receptor Agonist
5 PMC2946277_chunk0 0.0159 Effects of oral glucose-lowering drugs on long ter

=== weighted ===
1 PMC2644685 0.6 Metformin treatment in diabetes and heart failure:
2 PMC2991324 0.5377 Long-term effect of metformin on blood glucose con
3 PMC2946277_chunk0 0.5122 Effects of oral glucose-lowering drugs on lo

In [13]:
sys.path.append('../scripts')
rr_module = import_module('06_reranker')

reranker = rr_module.MultiCriteriaReranker()

test_candidates = [
    {"chunk_id": "test1", "text": "Metformin treatment in diabetes and heart failure patients."},
    {"chunk_id": "test2", "text": "The Soul's Wisdom: Stories of Living and Dying."},
]
results = reranker.rerank("What is the effect of metformin on cardiovascular disease?", test_candidates, top_k=2)
for r in results:
    print(r['rank'], r['chunk_id'], r['final_score'], r['relevance_score'])


加载reranker模型...


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 6834.71it/s]


1 test1 0.641 0.735
2 test2 0.2001 0.0001


In [15]:
importlib.reload(mpr_module)
importlib.reload(rr_module)
pipeline_module = import_module('07_retrieval_pipeline')

pipeline = pipeline_module.MedRAGPipeline(
    chunks_path='../data/processed/chunks.parquet',
    chroma_db_path='../data/processed/chroma_db',
    reranker_model_path='../bge-reranker-base-cache',
)

results = pipeline.run("What is the effect of metformin on cardiovascular disease?")
for r in results:
    print(r['rank'], r['chunk_id'], r['final_score'], r['text'][:60])


初始化 MultiPathRetriever...
加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 608.92it/s]


构建BM25索引...
初始化 Reranker...
加载reranker模型...


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 7026.75it/s]


1 PMC2946277_chunk0 0.3029 Effects of oral glucose-lowering drugs on long term outcomes
2 PMC2546413_chunk0 0.2297 Fatal hemolytic anemia associated with metformin: A case rep
3 PMC2566605_chunk0 0.2016 Effect of Adjunct Metformin Treatment in Patients with Type-
4 PMC2991324 0.1554 Long-term effect of metformin on blood glucose control in no
5 PMC1974811_chunk0 0.1277 Rosiglitazone RECORD study: glucose control outcomes at 18 m
6 PMC2664796_chunk0 0.1276 Diabetic cardiomyopathy: effects of fenofibrate and metformi
7 PMC2940872 0.1254 Therapies for type 2 diabetes: lowering HbA1c and associated
8 PMC2797799 0.124 Lifestyle modification and metformin as long-term treatment 
9 PMC2906460_chunk1 0.121 Conversely, pioglitazone had no impact on fasting insulin, t
10 PMC2169248_chunk0 0.1201 Metformin-induced lactic acidosis: a case series. Introducti
